In [1]:
import collections
import math
import random
import re

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained("sdadas/polish-gpt2-medium")
model = AutoModelForCausalLM.from_pretrained("sdadas/polish-gpt2-medium").to(device).eval()

/Users/bartek/Projects/courses-uwr/Modele Jezykowe/Pracownia2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1102 18:29:32.789000 98119 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 292/292 [00:00<00:00, 7149.16it/s]
[transformers] GPT2LMHeadModel LOAD REPORT from: sdadas/polish-gpt2-medium
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [2]:
def stems(text):
    return {t.lower()[:5] for t in re.findall(r"\w+", text)}


docs = collections.defaultdict(set)
with open("plwiktionary_for_riddles.txt", encoding="utf-8") as f:
    for line in f:
        word, definition = line.split(";;", 1)
        docs[word.strip()] |= stems(definition)

with open("zagadki_do_testow_clean.txt", encoding="utf-8") as f:
    riddles = [tuple(p.strip() for p in line.split(";;")) for line in f]

random.seed(0)
torch.manual_seed(0)
random.shuffle(riddles)
test = riddles[50:200]

## Zadanie 2

Słownik to drzewo (trie) po tokenach `" słowo"`. W węźle wolno wybrać tylko token-dziecko, a jeśli węzeł kończy słowo, także **koniec słowa** (`END`) o masie tokenów, które nie kontynuują wyrazu (to token nadmiarowy).

Żeby było szybko, nie losujemy `n` razy osobno: `n` próbek rozdzielamy między dzieci węzła, więc każdy prefiks liczymy tylko raz. Schodzimy po drzewie poziomami, a prefiksy z jednego poziomu mają tę samą długość, więc idą do modelu jednym batchem.

In [3]:
END = -1
boundary = torch.tensor(
    [i for i in range(len(tokenizer)) if not tokenizer.decode([i])[:1].isalpha()]
)


def build_trie(words):
    root = {}
    for word in words:
        node = root
        for token in tokenizer.encode(" " + word, add_special_tokens=False):
            node = node.setdefault(token, {})
        node[END] = word
    return root


@torch.no_grad()
def sample_words(prompt, n, trie):
    prompt_ids = tokenizer.encode(prompt)
    counts = collections.Counter()
    level = [(trie, [], n)]
    while level:
        batch = torch.tensor([prompt_ids + prefix for _, prefix, _ in level], device=device)
        logits = model(batch).logits[:, -1].cpu()
        end_logits = logits[:, boundary].logsumexp(-1, keepdim=True)
        logits = torch.cat([logits, end_logits], -1)  # ostatni indeks to END

        next_level = []
        for (node, prefix, count), node_logits in zip(level, logits):
            keys = list(node)
            probs = node_logits[keys].softmax(0)
            drawn = probs.multinomial(count, replacement=True).bincount(minlength=len(keys))
            for key, k in zip(keys, drawn.tolist()):
                if k and key == END:
                    counts[node[END]] += k
                elif k:
                    next_level.append((node[key], prefix + [key], k))
        level = next_level
    return counts


def make_prompt(riddle):
    return f"{riddle} Jedno słowo, które to opisuje, to:"


def report(name, rankings):
    ranks = [r.index(a) + 1 if a in r else None for (a, _), r in zip(test, rankings)]
    print(
        f"{name:<20} dokładność {ranks.count(1) / len(ranks):6.1%}, MRR {sum(1 / r for r in ranks if r) / len(ranks):.3f}"
    )


TRIE = build_trie(docs)
for n in [1, 20]:
    samples = [sample_words(make_prompt(riddle), n, TRIE) for _, riddle in test]
    assert all(w in docs for counts in samples for w in counts)
    report(
        f"LLM, n={n}", [[w for w, _ in counts.most_common(20)] for counts in samples]
    )

LLM, n=1             dokładność   0.7%, MRR 0.007
LLM, n=20            dokładność   6.0%, MRR 0.080


## Zadanie 3

TF-IDF po definicjach (rdzeń to 5 pierwszych liter) wybiera 100 kandydatów, a model generuje już tylko spośród nich. Końcowy wynik słowa to `log(1 + liczba wygenerowań) + α · TF-IDF`. Wagę `α = 1` dobrałem wcześniej na 50 zagadkach spoza zbioru testowego (`riddles[:50]`).

In [4]:
doc_freq = collections.Counter(stem for doc in docs.values() for stem in doc)


def tfidf(riddle):
    query = stems(riddle)
    scores = {
        w: sum(math.log(len(docs) / doc_freq[s]) for s in doc & query)
        / math.sqrt(len(doc))
        for w, doc in docs.items()
    }
    return dict(sorted(scores.items(), key=lambda x: -x[1])[:100])


def rank(candidates, counts, alpha):
    return sorted(
        candidates,
        key=lambda w: math.log1p(counts[w]) + alpha * candidates[w],
        reverse=True,
    )[:20]


candidates = [tfidf(riddle) for _, riddle in test]
samples = [
    sample_words(make_prompt(riddle), 20, build_trie(c))
    for (_, riddle), c in zip(test, candidates)
]

for name, alpha in [
    ("LLM na kandydatach", 0),
    ("samo TF-IDF", 1000),
    ("LLM + TF-IDF", 1),
]:
    report(name, [rank(c, counts, alpha) for c, counts in zip(candidates, samples)])

LLM na kandydatach   dokładność   9.3%, MRR 0.184
samo TF-IDF          dokładność  13.3%, MRR 0.204
LLM + TF-IDF         dokładność  17.3%, MRR 0.253
